# Stem separation (BS-RoFormer)

Pick an audio file with the popup, separate it into stems with **BS-RoFormer**, and export the WAVs
with normalized names in the project `stems/` folder, ready for the app to load.

**Minimum hardware requirements**

- RAM: 8 GB minimum (16 GB recommended)
- CPU: any modern x86_64 works; on CPU separation takes several minutes per track
- GPU: optional (CUDA speeds it up significantly)
- Disk: ~3 GB free (PyTorch + ~700 MB model, downloaded automatically on first run)
- Python ≥ 3.10

Usage: run the cells top to bottom (`Shift+Enter`).

In [1]:
# Install bs-roformer-infer only if it's missing
import importlib.util, subprocess, sys

if importlib.util.find_spec('bs_roformer_infer') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           'bs-roformer-infer @ git+https://github.com/openmirlab/bs-roformer-infer.git'])
print('bs-roformer-infer ready')

  Cloning https://github.com/openmirlab/bs-roformer-infer.git to /tmp/pip-install-qw7u_ws4/bs-roformer-infer_e5651e16839347568d455e840f469a37


  Running command git clone --filter=blob:none --quiet https://github.com/openmirlab/bs-roformer-infer.git /tmp/pip-install-qw7u_ws4/bs-roformer-infer_e5651e16839347568d455e840f469a37


  Resolved https://github.com/openmirlab/bs-roformer-infer.git to commit e3ad851fb24333ae36ae4bfd5ec327dbbf764bea
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
bs-roformer-infer ready


In [2]:
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()  # popup only, no main window
INPUT_FILE = filedialog.askopenfilename(
    title='Pick an audio file to separate',
    filetypes=[('Audio', '*.wav *.flac *.mp3 *.ogg *.m4a')])
root.destroy()

assert INPUT_FILE, 'No file selected'
print('Selected audio:', INPUT_FILE)

Selected audio: /home/lema/pas1dir/01-Mixer/stems/Proyecto Quasar - Prosopagnosia - 03 Placard.mp3


In [3]:
import shutil
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent          # from the notebook (stems/) to the project root
WORK = Path('input'); WORK.mkdir(exist_ok=True)   # temp input folder for the CLI
# The CLI only accepts WAV: convert the input if needed (ffmpeg, soundfile as fallback)
src = Path(INPUT_FILE)
wav_path = WORK / (src.stem + '.wav')
if src.suffix.lower() == '.wav':
    shutil.copy(src, wav_path)
else:
    try:
        subprocess.run(['ffmpeg', '-y', '-i', str(src), str(wav_path)], check=True, capture_output=True)
    except (FileNotFoundError, subprocess.CalledProcessError):
        import soundfile as sf
        data, rate = sf.read(src)
        sf.write(wav_path, data, rate)

subprocess.run(['bs-roformer-infer', '--input_folder', str(WORK),
                '--store_dir', str(WORK / 'out')], check=True)

# Normalize names: lowercase <stem>.wav, no spaces, in the project's stems/ folder
STEMS = ['vocals', 'drums', 'bass', 'guitar', 'piano', 'other', 'instrumental']
exported = []
for f in (WORK / 'out').glob('*.wav'):
    label = next((s for s in STEMS if s in f.stem.lower()), None)
    if label:
        dest = PROJECT_ROOT / 'stems' / f'{label}.wav'
        shutil.move(str(f), dest)
        exported.append(dest.name)

shutil.rmtree(WORK)  # clean up temp files
print('Stems exported to', PROJECT_ROOT / 'stems', ':', sorted(exported))

Using model weights: /home/lema/.cache/bs-roformer-infer/roformer-model-bs-roformer-sw-by-jarredou/BS-Rofo-SW-Fixed.ckpt
Non-A100 GPU detected, using math or mem efficient attention if input tensor is on cuda


/home/lema/miniconda3/envs/esaad_pas1/lib/python3.11/site-packages/torch/functional.py:680: UserWarning: A window was not provided. A rectangular window will be applied,which is known to cause spectral leakage. Other windows such as torch.hann_window or torch.hamming_window are recommended to reduce spectral leakage.To suppress this warning and use a rectangular window, explicitly set `window=torch.ones(n_fft, device=<device>)`. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/SpectralOps.cpp:855.)
  return _VF.stft(  # type: ignore[attr-defined]


Total tracks found: 1

Processing track 1/1: Proyecto Quasar - Prosopagnosia - 03 Placard.wav


Tracks:   0%|          | 0/1 [00:00<?, ?track/s]/home/lema/miniconda3/envs/esaad_pas1/lib/python3.11/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Estimated total processing time for this track: 67.83 seconds
Estimated time remaining: 0.00 secondss


Tracks: 100%|██████████| 1/1 [00:47<00:00, 47.65s/track]


Elapsed time: 48.66 sec
Stems exported to /home/lema/pas1dir/01-Mixer : ['bass.wav', 'drums.wav', 'guitar.wav', 'instrumental.wav', 'other.wav', 'piano.wav', 'vocals.wav']
